# Data Cleaning Report
## Network Traffic Dataset Cleaning Pipeline

**Project**: Real-Time Network Traffic Classifier

**Purpose**: Clean raw network traffic CSV files for machine learning and analytics

**Objectives**:
- Remove duplicates and corrupted records
- Handle missing values intelligently
- Normalize data types and formats
- Extract temporal features for time-based queries
- Remove statistical outliers
- Prepare data for flow-based analysis and classification

**Input Folder**: `Extracted Data/`

**Output Folder**: `Cleaned Data/`

## Setup & Configuration

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Pandas display options
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 150)
pd.set_option('display.max_colwidth', 30)

print('Notebook started at:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

Notebook started at: 2025-11-13 18:40:20


In [2]:
# Detect project root and set paths
cwd = Path.cwd()
project_root = cwd
while not (project_root / 'Extracted Data').exists() and project_root != project_root.parent:
    project_root = project_root.parent

if not (project_root / 'Extracted Data').exists():
    raise FileNotFoundError("Could not find 'Extracted Data' folder")

input_dir = project_root / 'Extracted Data'
output_dir = project_root / 'Cleaned Data'
output_dir.mkdir(exist_ok=True)

print(f'Project Root: {project_root}')
print(f'Input Directory: {input_dir}')
print(f'Output Directory: {output_dir}')

# List all CSV files
csv_files = sorted([p for p in input_dir.glob('*.csv')])
print(f'\nFound {len(csv_files)} CSV files:')
for i, f in enumerate(csv_files, 1):
    size_mb = f.stat().st_size / (1024*1024)
    print(f'  {i}. {f.name:50s} ({size_mb:8.2f} MB)')

Project Root: /Users/divyanshioberoi/Desktop/IIT/Fall 2025/CS597/RealTime-Network-Traffic-Classifier
Input Directory: /Users/divyanshioberoi/Desktop/IIT/Fall 2025/CS597/RealTime-Network-Traffic-Classifier/Extracted Data
Output Directory: /Users/divyanshioberoi/Desktop/IIT/Fall 2025/CS597/RealTime-Network-Traffic-Classifier/Cleaned Data

Found 1 CSV files:
  1. Monday_pcap_to_csv_data.csv                        ( 1773.24 MB)


## Helper Functions

In [3]:
def print_section(title):
    """Print a formatted section header"""
    print('\n' + '='*80)
    print(f'  {title}')
    print('='*80)

def print_before_after(before, after, step_name):
    """Print before/after comparison"""
    removed = before - after
    pct = (removed / before * 100) if before > 0 else 0
    print(f'{step_name}:')
    print(f'  Before: {before:>12,} rows')
    print(f'  After:  {after:>12,} rows')
    print(f'  Removed: {removed:>11,} rows ({pct:5.2f}%)')
    return removed

def get_data_summary(df, filename):
    """Get comprehensive data summary"""
    return {
        'filename': filename,
        'rows': len(df),
        'columns': len(df.columns),
        'memory_mb': df.memory_usage(deep=True).sum() / (1024*1024),
        'null_count': df.isnull().sum().sum(),
        'dtypes': df.dtypes.value_counts().to_dict()
    }

print('Helper functions loaded')

Helper functions loaded


## Processing Summary Table (Before All Files)

This table shows statistics for all files BEFORE cleaning:

In [4]:
print_section('ORIGINAL FILES STATISTICS')
original_stats = []

for csv_file in csv_files:
    try:
        df_temp = pd.read_csv(csv_file, low_memory=False)
        stats = get_data_summary(df_temp, csv_file.name)
        original_stats.append(stats)
        
        print(f'\n{csv_file.name}')
        print(f'  Rows: {stats["rows"]:,}')
        print(f'  Columns: {stats["columns"]}')
        print(f'  Memory: {stats["memory_mb"]:.2f} MB')
        print(f'  Null Values: {stats["null_count"]:,}')
        print(f'  Dtype Distribution: {stats["dtypes"]}')
    except Exception as e:
        print(f'Error reading {csv_file.name}: {e}')

print_section('ORIGINAL FILES SUMMARY')
original_df = pd.DataFrame(original_stats)
display(original_df[['filename', 'rows', 'columns', 'memory_mb', 'null_count']])


  ORIGINAL FILES STATISTICS

Monday_pcap_to_csv_data.csv
  Rows: 11,709,971
  Columns: 8
  Memory: 3763.16 MB
  Null Values: 57,150
  Dtype Distribution: {dtype('O'): 4, dtype('int64'): 2, dtype('float64'): 2}

  ORIGINAL FILES SUMMARY


,filename,rows,columns,memory_mb,null_count
0,Monday_pcap_to_csv_data.csv,11709971,8,3763.158508,57150


## Data Cleaning Pipeline

Processing each file through the following steps:

1. **Load Data**: Read CSV file
2. **Remove Duplicates**: Eliminate exact row duplicates
3. **Handle Missing Values**: Fill or drop rows with missing data
4. **Fix Data Types**: Parse timestamps, convert numeric columns
5. **Extract Time Features**: Create day-of-week and date columns
6. **Filter Irrelevant Traffic**: Remove broadcast/multicast/zero-packet rows
7. **Normalize Values**: Standardize IPs, ports, protocols
8. **Remove Outliers**: Use IQR method to identify and remove extreme values
9. **Save Cleaned Data**: Write to output folder

In [5]:
def clean_single_file(input_path, output_path):
    """
    Complete cleaning pipeline for a single file.
    Returns: (original_df_shape, final_df_shape, detailed_results)
    """
    results = {
        'filename': input_path.name,
        'original_shape': None,
        'final_shape': None,
        'steps': {}
    }
    
    print_section(f'CLEANING: {input_path.name}')
    
    # Load
    df = pd.read_csv(input_path, low_memory=False)
    original_rows = len(df)
    original_cols = len(df.columns)
    results['original_shape'] = (original_rows, original_cols)
    print(f'\nLoaded: {original_rows:,} rows × {original_cols} columns')
    print(f'Memory: {df.memory_usage(deep=True).sum() / (1024*1024):.2f} MB')
    
    # Step 1: Duplicates
    print('\n[Step 1] Removing Duplicates')
    before = len(df)
    df = df.drop_duplicates()
    removed = print_before_after(before, len(df), 'Deduplication')
    results['steps']['duplicates'] = removed
    
    # Step 2: Handle missing values
    print('\n[Step 2] Handling Missing & Infinite Values')
    null_before = df.isnull().sum().sum()
    print(f'Null values before: {null_before:,}')
    
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    for col in numeric_cols:
        if df[col].isnull().any():
            if any(x in col.lower() for x in ['packet', 'pkt', 'byte', 'len']):
                df[col] = df[col].fillna(0)
            else:
                df[col] = df[col].fillna(df[col].median())
    
    cat_cols = df.select_dtypes(include=['object']).columns.tolist()
    for col in cat_cols:
        if df[col].isnull().any():
            try:
                df[col] = df[col].fillna(df[col].mode()[0])
            except Exception:
                df[col] = df[col].fillna('')
    
    null_after = df.isnull().sum().sum()
    print(f'Null values after: {null_after:,}')
    results['steps']['missing_values'] = null_before - null_after
    
    # Step 3: Data types & time features
    print('\n[Step 3] Converting Data Types & Extracting Time Features')
    cols_added = 0
    timestamp_cols = [c for c in df.columns if 'time' in c.lower() or 'timestamp' in c.lower()]
    for col in timestamp_cols:
        try:
            df[col] = pd.to_datetime(df[col])
            df[f'{col}_dayofweek'] = df[col].dt.day_name()
            df[f'{col}_dayofweek_int'] = df[col].dt.dayofweek
            df[f'{col}_date'] = df[col].dt.date
            cols_added += 3
            print(f'  Parsed timestamp column: {col}')
        except Exception as e:
            print(f'  Could not parse {col}: {e}')
    
    print(f'Added {cols_added} time-related columns')
    results['steps']['time_columns_added'] = cols_added
    
    # Step 4: Normalize strings
    print('\n[Step 4] Normalizing Column Names & String Values')
    df.columns = [c.lower().strip() for c in df.columns]
    for col in df.select_dtypes(include=['object', 'string']).columns:
        try:
            df[col] = df[col].astype('string').str.strip().str.lower()
        except Exception:
            pass
    print('  Columns normalized to lowercase')
    
    # Step 5: Convert ports
    print('\n[Step 5] Converting Port Columns to Integer')
    port_cols = [c for c in df.columns if 'port' in c]
    for c in port_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce').astype('Int64')
    print(f'  Converted {len(port_cols)} port columns')
    
    # Step 6: Filter broadcast/multicast/zero-packet
    print('\n[Step 6] Filtering Irrelevant Traffic')
    before = len(df)
    
    pkt_cols = [c for c in df.columns if any(k in c for k in ['packet','pkt','len','size','byte','byts'])]
    combined_mask = pd.Series(False, index=df.index)
    
    # Broadcast
    for ip_col in [c for c in df.columns if 'dst' in c and 'ip' in c]:
        s = df[ip_col].fillna('').astype(str).str.strip()
        combined_mask = combined_mask | (s == '255.255.255.255')
        # Multicast
        first_oct = s.str.split('.', expand=True)[0]
        combined_mask = combined_mask | first_oct.map(lambda x: str(x).isdigit() and 224 <= int(x) <= 239)
    
    # Zero packets
    mask_zero = pd.Series(False, index=df.index)
    for c in pkt_cols:
        pktnum = pd.to_numeric(df[c], errors='coerce')
        mask_zero = mask_zero | pktnum.isna() | (pktnum == 0)
    
    combined_mask = combined_mask | mask_zero
    
    if combined_mask.sum() < len(df):
        df = df.loc[~combined_mask].copy()
        removed = print_before_after(before, len(df), 'Traffic Filtering')
        results['steps']['traffic_filtering'] = removed
    else:
        print('  No rows removed (would remove all data)')
        results['steps']['traffic_filtering'] = 0
    
    # Step 7: Remove outliers
    print('\n[Step 7] Removing Statistical Outliers (IQR Method)')
    before = len(df)
    numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if 'port' not in c]
    outlier_count = 0
    
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        if pd.isna(IQR) or IQR == 0:
            continue
        lower, upper = Q1 - 3*IQR, Q3 + 3*IQR
        old_len = len(df)
        df = df[(df[col] >= lower) & (df[col] <= upper)]
        removed_this = old_len - len(df)
        if removed_this > 0:
            outlier_count += removed_this
    
    removed = print_before_after(before, len(df), 'Outlier Removal')
    results['steps']['outliers'] = removed
    
    # Step 8: Save
    print('\n[Step 8] Saving Cleaned File')
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    final_rows = len(df)
    final_cols = len(df.columns)
    final_memory = df.memory_usage(deep=True).sum() / (1024*1024)
    
    results['final_shape'] = (final_rows, final_cols)
    
    print(f'Saved: {output_path.name}')
    print(f'  Final: {final_rows:,} rows × {final_cols} columns')
    print(f'  Memory: {final_memory:.2f} MB')
    
    return results

print('Pipeline function defined')

Pipeline function defined


## Run Cleaning Pipeline on All Files

In [6]:
all_results = []

for csv_file in csv_files:
    try:
        output_file = output_dir / f'cleaned_{csv_file.name}'
        result = clean_single_file(csv_file, output_file)
        all_results.append(result)
    except Exception as e:
        print(f'\nERROR processing {csv_file.name}: {e}')
        import traceback
        traceback.print_exc()

print_section('CLEANING PIPELINE COMPLETED')
print(f'Successfully processed {len(all_results)} files')


  CLEANING: Monday_pcap_to_csv_data.csv

Loaded: 11,709,971 rows × 8 columns
Memory: 3763.16 MB

[Step 1] Removing Duplicates
Deduplication:
  Before:   11,709,971 rows
  After:    11,709,971 rows
  Removed:           0 rows ( 0.00%)

[Step 2] Handling Missing & Infinite Values
Null values before: 57,150
Null values after: 0

[Step 3] Converting Data Types & Extracting Time Features
  Parsed timestamp column: Time
Added 3 time-related columns

[Step 4] Normalizing Column Names & String Values
  Columns normalized to lowercase

[Step 5] Converting Port Columns to Integer
  Converted 1 port columns

[Step 6] Filtering Irrelevant Traffic
Traffic Filtering:
  Before:   11,709,971 rows
  After:    11,709,971 rows
  Removed:           0 rows ( 0.00%)

[Step 7] Removing Statistical Outliers (IQR Method)
Outlier Removal:
  Before:   11,709,971 rows
  After:    11,678,092 rows
  Removed:      31,879 rows ( 0.27%)

[Step 8] Saving Cleaned File
Saved: cleaned_Monday_pcap_to_csv_data.csv
  Final:

## Final Summary Report

### Before & After Comparison

In [7]:
print_section('COMPREHENSIVE CLEANING SUMMARY')

summary_data = []
for result in all_results:
    orig_rows, orig_cols = result['original_shape']
    final_rows, final_cols = result['final_shape']
    rows_removed = orig_rows - final_rows
    rows_removed_pct = (rows_removed / orig_rows * 100) if orig_rows > 0 else 0
    
    summary_data.append({
        'File': result['filename'],
        'Original Rows': f"{orig_rows:,}",
        'Cleaned Rows': f"{final_rows:,}",
        'Rows Removed': f"{rows_removed:,}",
        'Removal %': f"{rows_removed_pct:.2f}%",
        'Columns': f"{orig_cols} → {final_cols}",
        'Duplicates': f"{result['steps'].get('duplicates', 0):,}",
        'Nulls Fixed': f"{result['steps'].get('missing_values', 0):,}",
        'Outliers': f"{result['steps'].get('outliers', 0):,}"
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df)


  COMPREHENSIVE CLEANING SUMMARY


,File,Original Rows,Cleaned Rows,Rows Removed,Removal %,Columns,Duplicates,Nulls Fixed,Outliers
0,Monday_pcap_to_csv_data.csv,"11,709,971","11,678,092","31,879",0.27%,8 → 11,0,"57,150","31,879"


### Key Metrics

In [8]:
print_section('KEY CLEANING METRICS')

total_original_rows = sum(r['original_shape'][0] for r in all_results)
total_final_rows = sum(r['final_shape'][0] for r in all_results)
total_rows_removed = total_original_rows - total_final_rows
total_removal_pct = (total_rows_removed / total_original_rows * 100) if total_original_rows > 0 else 0

total_duplicates = sum(r['steps'].get('duplicates', 0) for r in all_results)
total_nulls_fixed = sum(r['steps'].get('missing_values', 0) for r in all_results)
total_outliers = sum(r['steps'].get('outliers', 0) for r in all_results)
total_traffic_filtered = sum(r['steps'].get('traffic_filtering', 0) for r in all_results)

print(f'\nTotal Files Processed: {len(all_results)}')
print(f'\nRows:')
print(f'  Original:  {total_original_rows:>15,}')
print(f'  Cleaned:   {total_final_rows:>15,}')
print(f'  Removed:   {total_rows_removed:>15,} ({total_removal_pct:.2f}%)')
print(f'\nCleaning Actions:')
print(f'  Duplicates Removed:      {total_duplicates:>15,}')
print(f'  Null Values Fixed:       {total_nulls_fixed:>15,}')
print(f'  Irrelevant Traffic:      {total_traffic_filtered:>15,}')
print(f'  Outliers Removed:        {total_outliers:>15,}')


  KEY CLEANING METRICS

Total Files Processed: 1

Rows:
  Original:       11,709,971
  Cleaned:        11,678,092
  Removed:            31,879 (0.27%)

Cleaning Actions:
  Duplicates Removed:                    0
  Null Values Fixed:                57,150
  Irrelevant Traffic:                    0
  Outliers Removed:                 31,879


### Cleaned Files Location

In [9]:
print_section('CLEANED FILES')

cleaned_files = sorted([p for p in output_dir.glob('cleaned_*.csv')])
print(f'\nTotal cleaned files: {len(cleaned_files)}')
print(f'\nLocation: {output_dir}')
print('\nFiles:')
for f in cleaned_files:
    size_mb = f.stat().st_size / (1024*1024)
    print(f'  ✓ {f.name:50s} ({size_mb:8.2f} MB)')

print(f'\nCompleted at: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')


  CLEANED FILES

Total cleaned files: 1

Location: /Users/divyanshioberoi/Desktop/IIT/Fall 2025/CS597/RealTime-Network-Traffic-Classifier/Cleaned Data

Files:
  ✓ cleaned_Monday_pcap_to_csv_data.csv                ( 2022.18 MB)

Completed at: 2025-11-13 18:44:55


### Data Quality Checks

In [10]:
print_section('DATA QUALITY VERIFICATION')

for cleaned_file in cleaned_files:
    df_check = pd.read_csv(cleaned_file, nrows=1000, low_memory=False)
    print(f'\n{cleaned_file.name}')
    print(f'  Null values: {df_check.isnull().sum().sum()}')
    print(f'  Infinite values: {np.isinf(df_check.select_dtypes(include=[np.number])).sum().sum()}')
    print(f'  Duplicates (sample): {df_check.duplicated().sum()}')
    print(f'  ✓ Quality checks passed')


  DATA QUALITY VERIFICATION

cleaned_Monday_pcap_to_csv_data.csv
  Null values: 0
  Infinite values: 0
  Duplicates (sample): 0
  ✓ Quality checks passed


## Next Steps

✓ Cleaned data is ready for:
- Feature extraction and engineering
- Machine learning model training
- Database ingestion
- Analytics and visualization
- Time-based querying (using timestamp_dayofweek and timestamp_date columns)